# 🧠 A Armadilha das Matrizes em Python
### Objetos Mutáveis, Ponteiros e Aliasing de Memória

**Disciplina:** Estruturas de Dados (IBM3114) | **Professor:** Luís Aramis

**Objetivo:** Compreender por que criar matrizes com o operador `*` gera um comportamento inesperado, aprendendo os conceitos de **objeto mutável**, **ponteiro** e **aliasing** de memória.

**Aluno:** [Lucas Monteiro]

---

## 🚨 Passo 1: Reproduzindo o Bug

Execute a célula abaixo e tente **prever o resultado antes de rodar**.

> *O código tenta criar um tabuleiro 3×3 e marcar apenas a posição `[0][0]` com o valor `1`.*

In [2]:
# Criação "ingênua" de uma matriz 3x3 usando o operador *
tabuleiro = [[0] * 3] * 3


In [11]:

tabuleiro = [[0 for _ in range(3)] for _ in range(3)]

tabuleiro[0][0] = 1
for linha in tabuleiro:
    print(linha) 

[1, 0, 0]
[0, 0, 0]
[0, 0, 0]


### 🤔 Peer Instruction — Discuta com um colega:

- Quantas células foram alteradas? Era o esperado?
- Por que `tabuleiro[1][0]` e `tabuleiro[2][0]` também mudaram se só tocamos em `tabuleiro[0][0]`?

---

## 🔬 Passo 2: Diagnóstico — O que `id()` revela?

Em Python, a função `id()` retorna o **endereço de memória** de um objeto.
Se duas variáveis têm o mesmo `id`, elas apontam para o **mesmo objeto físico** na RAM.

> 💡 Pense no `id()` como o número do apartamento de um objeto na memória do computador.

In [4]:
# Recriando o tabuleiro para partir de um estado limpo
tabuleiro = [[0] * 3] * 3

print("Endereços de memória (id) de cada linha:")
for i, linha in enumerate(tabuleiro):
    print(f"  tabuleiro[{i}] → id = {id(linha)}")

print()

# Todas as linhas são o MESMO objeto?
sao_iguais = (id(tabuleiro[0]) == id(tabuleiro[1]) == id(tabuleiro[2]))
print(f"Todas as linhas são o mesmo objeto? → {sao_iguais}")

Endereços de memória (id) de cada linha:
  tabuleiro[0] → id = 140370941839296
  tabuleiro[1] → id = 140370941839296
  tabuleiro[2] → id = 140370941839296

Todas as linhas são o mesmo objeto? → True


### O que está acontecendo na memória?

```
Quando você escreve  [[0] * 3] * 3,  o Python faz:

  Passo 1: cria UMA lista interna:    [0, 0, 0]   ← Objeto A (endereço: 0x5F3A)
  Passo 2: cria a lista externa com 3 REFERÊNCIAS para o mesmo Objeto A:

  tabuleiro = [ 0x5F3A, 0x5F3A, 0x5F3A ]
                  ↓        ↓        ↓
               [0,0,0]  [0,0,0]  [0,0,0]   ← são o MESMO objeto!

  Ao fazer tabuleiro[0][0] = 1,  você altera o Objeto A.
  Como todas as linhas apontam para ele, TODAS aparecem modificadas.
```

---

## 🧩 Passo 3: Entendendo os Conceitos-Chave

### Objeto Mutável
Listas em Python são **mutáveis**: você pode alterar seu conteúdo sem criar uma nova lista.
Compare com **strings** (imutáveis): ao "alterar" uma string, Python cria um novo objeto.

In [5]:
# Demonstrando mutabilidade: lista vs string

# Lista (mutável): a e b apontam para o MESMO objeto
a = [1, 2, 3]
b = a               # b é um APELIDO de a, não uma cópia!
b[0] = 99
print(f"Lista mutável:   a = {a}  ← a também mudou!")
print(f"                 b = {b}")
print(f"                 id(a) == id(b)? {id(a) == id(b)}\n")

# String (imutável): x e y são independentes após a 're-atribuição'
x = "dados"
y = x
y = y + "_novos"    # cria um NOVO objeto, não modifica o original
print(f"String imutável: x = '{x}'  ← x NÃO mudou")
print(f"                 y = '{y}'")
print(f"                 id(x) == id(y)? {id(x) == id(y)}")

Lista mutável:   a = [99, 2, 3]  ← a também mudou!
                 b = [99, 2, 3]
                 id(a) == id(b)? True

String imutável: x = 'dados'  ← x NÃO mudou
                 y = 'dados_novos'
                 id(x) == id(y)? False


### Ponteiro / Referência

Em Python, variáveis **não guardam valores diretamente** — elas guardam **referências** (endereços) para objetos na memória.
O operador `*` em listas copia essas referências, não os objetos em si.

In [6]:
# Visualizando referências com uma lista simples
linha_unica = [0, 0, 0]
matriz_errada = [linha_unica] * 3   # copia a REFERÊNCIA 3 vezes

print("Antes da modificação:")
print(f"  linha_unica:  {linha_unica}   id={id(linha_unica)}")
for i, row in enumerate(matriz_errada):
    print(f"  matriz[{i}]:    {row}   id={id(row)}")

# Modificar linha_unica afeta TODA a matriz!
linha_unica[1] = 7

print("\nApós linha_unica[1] = 7:")
print(f"  linha_unica:  {linha_unica}")
for i, row in enumerate(matriz_errada):
    print(f"  matriz[{i}]:    {row}")

Antes da modificação:
  linha_unica:  [0, 0, 0]   id=140370941852288
  matriz[0]:    [0, 0, 0]   id=140370941852288
  matriz[1]:    [0, 0, 0]   id=140370941852288
  matriz[2]:    [0, 0, 0]   id=140370941852288

Após linha_unica[1] = 7:
  linha_unica:  [0, 7, 0]
  matriz[0]:    [0, 7, 0]
  matriz[1]:    [0, 7, 0]
  matriz[2]:    [0, 7, 0]


---

## ✅ Passo 4: A Solução Correta — List Comprehension

A solução é garantir que **cada linha seja um objeto independente** na memória.
O `for` da list comprehension chama `[0] * colunas` **a cada iteração**, criando uma nova lista a cada vez.

In [7]:
def criar_matriz_segura(linhas: int, colunas: int, valor: int = 0) -> list[list[int]]:
    """
    Cria uma matriz (linhas x colunas) com objetos INDEPENDENTES por linha.
    Usa list comprehension para garantir que cada linha seja um novo objeto.
    Complexidade: O(linhas * colunas).
    """
    return [[valor] * colunas for _ in range(linhas)]


tabuleiro_safe = criar_matriz_segura(3, 3)

print("Endereços de cada linha (list comprehension):")
for i, linha in enumerate(tabuleiro_safe):
    print(f"  tabuleiro_safe[{i}] → id = {id(linha)}")

sao_iguais = (id(tabuleiro_safe[0]) == id(tabuleiro_safe[1]))
print(f"\nLinhas são o mesmo objeto? → {sao_iguais}  ✅ Correto!")

Endereços de cada linha (list comprehension):
  tabuleiro_safe[0] → id = 140370515572352
  tabuleiro_safe[1] → id = 140370515570688
  tabuleiro_safe[2] → id = 140370515570304

Linhas são o mesmo objeto? → False  ✅ Correto!


In [8]:
# Confirmando: agora apenas [0][0] muda
tabuleiro_safe[0][0] = 1

print("Após tabuleiro_safe[0][0] = 1:")
for linha in tabuleiro_safe:
    print(" ", linha)

print("\n✅ Apenas a posição [0][0] foi alterada!")

Após tabuleiro_safe[0][0] = 1:
  [1, 0, 0]
  [0, 0, 0]
  [0, 0, 0]

✅ Apenas a posição [0][0] foi alterada!


---

## ⚖️ Passo 5: Comparação Lado a Lado

In [ ]:
print("=" * 48)
print("  ERRADO: [[0]*3] * 3  vs  CORRETO: list comprehension")
print("=" * 48)

errado  = [[0] * 3] * 3
correto = [[0] * 3 for _ in range(3)]

# Marcando [1][2] em ambos
errado[1][2]  = 9
correto[1][2] = 9

print("\nApós marcar [1][2] = 9:")
print("\n  ❌ Errado (aliasing):")
for row in errado:
    print("    ", row)

print("\n  ✅ Correto (list comprehension):")
for row in correto:
    print("    ", row)

  ERRADO: [[0]*3] * 3  vs  CORRETO: list comprehension

Após marcar [1][2] = 9:

  ❌ Errado (aliasing):
     [0, 0, 9]
     [0, 0, 9]
     [0, 0, 9]

  ✅ Correto (list comprehension):
     [0, 0, 0]
     [0, 0, 9]
     [0, 0, 0]


---

## 📝 Conclusão

*(O aluno deve preencher com suas palavras:)*

1. **O bug:** Explique por que `[[0] * 3] * 3` não cria três linhas independentes. O que o operador `*` faz com objetos mutáveis?

2. **O diagnóstico:** Como a função `id()` ajudou a confirmar o problema de aliasing?

3. **A solução:** Por que a list comprehension `[[0] * 3 for _ in range(3)]` resolve o problema? O que acontece de diferente na memória a cada iteração do `for`?